# PySpark Silver Layer Analysis

Using PySpark to analyze stock data

In [1]:
# Add backend to path
import sys
sys.path.insert(0, r'D:\School\kltn\stock-lakehouse-ai\backend')

In [2]:
# Load data with pandas first, then convert to Spark
import pandas as pd

from app.lakehouse import SilverLayer
silver = SilverLayer()

# Load AAPL data
df_pandas = silver.read('AAPL')
print(f'Loaded {len(df_pandas)} records')
print(df_pandas.head())

PROJECT_ROOT: D:\School\kltn\stock-lakehouse-ai
BACKEND_ROOT: D:\School\kltn\stock-lakehouse-ai\backend
Env 1 exists: True
Env 2 exists: True


2026-09-29 16:10:14 | INFO | app.lakehouse.storage_factory | Using MinIO storage backend.


Loaded 1253 records
  symbol                 timestamp        open        high         low  \
0   AAPL 2021-09-30 13:30:00+00:00  143.660004  144.380005  141.279999   
1   AAPL 2021-10-01 13:30:00+00:00  141.899994  142.919998  139.110001   
2   AAPL 2021-10-04 13:30:00+00:00  141.759995  142.210007  138.270004   
3   AAPL 2021-10-05 13:30:00+00:00  139.490005  142.240005  139.360001   
4   AAPL 2021-10-06 13:30:00+00:00  139.470001  142.149994  138.369995   

        close   adj_close      volume    source  \
0  141.500000  141.500000  89056700.0  yfinance   
1  142.649994  142.649994  94639600.0  yfinance   
2  139.139999  139.139999  98322000.0  yfinance   
3  141.110001  141.110001  80861100.0  yfinance   
4  142.000000  142.000000  83221100.0  yfinance   

                    ingestion_time  ...  bb_position  atr_14  volume_sma_20  \
0 2026-09-29 07:39:05.217092+00:00  ...          NaN     NaN            NaN   
1 2026-09-29 07:39:05.217092+00:00  ...          NaN     NaN          

In [3]:
# Create Spark session - simple config
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('StockLakehouse-PySpark') \
    .master('local[4]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark Version:', spark.version)

Py4JError: An error occurred while calling None.org.apache.spark.sql.SparkSession. Trace:
py4j.Py4JException: Constructor org.apache.spark.sql.SparkSession([class org.apache.spark.SparkContext, class java.util.HashMap]) does not exist
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:180)
	at py4j.reflection.ReflectionEngine.getConstructor(ReflectionEngine.java:197)
	at py4j.Gateway.invoke(Gateway.java:237)
	at py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
	at py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:184)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:108)
	at java.base/java.lang.Thread.run(Thread.java:842)



In [ ]:
# Convert pandas to Spark DataFrame
df_spark = spark.createDataFrame(df_pandas)
df_spark.createOrReplaceTempView('stocks')

df_spark.printSchema()

In [ ]:
# Spark SQL Analysis
spark.sql('''
SELECT 
    COUNT(*) as total_days,
    MIN(timestamp) as start_date,
    MAX(timestamp) as end_date,
    ROUND(MIN(close), 2) as min_price,
    ROUND(MAX(close), 2) as max_price,
    ROUND(AVG(close), 2) as avg_price
FROM stocks
''').show()

In [ ]:
# Returns analysis
spark.sql('''
SELECT 
    symbol,
    COUNT(*) as trading_days,
    ROUND(AVG(return_1d) * 100, 4) as avg_daily_return_pct,
    ROUND(STDDEV(return_1d) * 100, 4) as std_daily_return_pct,
    ROUND(MIN(return_1d) * 100, 2) as worst_day_pct,
    ROUND(MAX(return_1d) * 100, 2) as best_day_pct
FROM stocks
WHERE return_1d IS NOT NULL
GROUP BY symbol
''').show(truncate=False)

In [ ]:
# Technical Indicators
spark.sql('''
SELECT 
    symbol,
    ROUND(AVG(rsi_14), 1) as avg_rsi,
    ROUND(AVG(macd), 3) as avg_macd,
    ROUND(AVG(volume_ratio), 2) as avg_volume_ratio,
    ROUND(AVG(volatility_20d) * 100, 2) as vol_pct
FROM stocks
WHERE rsi_14 IS NOT NULL
GROUP BY symbol
''').show(truncate=False)

In [ ]:
# Window functions for moving averages
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_20 = Window.orderBy('timestamp').rowsBetween(-20, 0)

df_with_ma = df_spark.withColumn(
    'sma_20_spark',
    F.avg('close').over(window_20)
)

df_with_ma.select(
    'timestamp', 'close', 'sma_20', 'sma_20_spark'
).filter('sma_20 IS NOT NULL').show(10, truncate=False)

In [ ]:
# Load multiple symbols
from app.lakehouse.storage_factory import get_storage_backend

storage = get_storage_backend()
objects = storage.list_objects('silver', '')

symbols = sorted(set([
    obj.replace('\\\\', '/').split('/')[0].replace('symbol=', '')
    for obj in objects if obj.startswith('symbol=')
]))[:5]

combined = None
for sym in symbols:
    sym_df = silver.read(sym)
    if not sym_df.empty:
        sdf = spark.createDataFrame(sym_df)
        combined = sdf if combined is None else combined.union(sdf)

combined.createOrReplaceTempView('multi_stocks')

spark.sql('''
SELECT 
    symbol,
    COUNT(*) as days,
    ROUND(AVG(rsi_14), 1) as avg_rsi
FROM multi_stocks
WHERE rsi_14 IS NOT NULL
GROUP BY symbol
''').show()